# Geometric EEG SSL — Data Download Notebook (CPU only)

**Purpose:** Cache all three datasets to Google Drive once. No GPU needed —
**do not waste GPU credit running this**. Use a standard CPU runtime.

After this notebook completes:
- PhysioNet MI is enough to start **`colab_pretrain.ipynb`** (all 5 variants).
- BCIC-2B and Sleep-EDFx are only required to run **`colab_experiment.ipynb`**.

Sections **4a / 4b / 4c** are independent — you can run them in any order or
re-run individually. All three are resume-aware: already-cached files are skipped.

---

## 1. Install dependencies

In [1]:
%%capture
!pip install mne moabb scikit-learn pyyaml scipy

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data cached here — avoids re-downloading across sessions
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.makedirs(MNE_DATA_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Checkpoints saved here
CKPT_ROOT = f'{DRIVE_ROOT}/runs'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Drive mounted. Checkpoints → {CKPT_ROOT}')
# Preprocessing cache (post-bandpass, resample, epoch, normalize).
# Set so dataset loaders cache to Drive — built once here, reused
# by every pretrain + eval run.
CACHE_ROOT = f'{DRIVE_ROOT}/cache'
os.makedirs(CACHE_ROOT, exist_ok=True)
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
print(f'Cache dir → {CACHE_ROOT}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Checkpoints → /content/drive/MyDrive/geometric_eeg_ssl/runs
Cache dir → /content/drive/MyDrive/geometric_eeg_ssl/cache


## 3. Clone repo (optional, for the loader code)

In [3]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR)

Already up to date.
Repo ready at /content/geometric-eeg-ssl


## 4a. Download PhysioNet MI data

In [4]:
import os, mne
from concurrent.futures import ThreadPoolExecutor, as_completed

mne.set_log_level('WARNING')

EXCLUDED = {88, 92, 100, 104}
ALL_SUBJECTS_MI = [s for s in range(1, 110) if s not in EXCLUDED]  # 105 subjects
MI_RUNS = [4, 6, 8, 10, 12, 14]

EEGBCI_ROOT = os.path.join(MNE_DATA_DIR, 'MNE-eegbci-data', 'files', 'eegmmidb', '1.0.0')

def subject_fully_cached(subj, runs):
    subj_dir = os.path.join(EEGBCI_ROOT, f'S{subj:03d}')
    if not os.path.isdir(subj_dir):
        return False
    return all(
        os.path.exists(os.path.join(subj_dir, f'S{subj:03d}R{run:02d}.edf')) and
        os.path.getsize(os.path.join(subj_dir, f'S{subj:03d}R{run:02d}.edf')) > 0
        for run in runs
    )

cached = [s for s in ALL_SUBJECTS_MI if subject_fully_cached(s, MI_RUNS)]
todo = [s for s in ALL_SUBJECTS_MI if s not in cached]
print(f'PhysioNet MI: cached {len(cached)}/{len(ALL_SUBJECTS_MI)} subjects. '
      f'Downloading {len(todo)} in parallel...')

N_WORKERS = 8  # I/O-bound; pooch atomic-moves make concurrent writes safe

def _fetch_one(subj):
    try:
        mne.datasets.eegbci.load_data(subj, MI_RUNS, path=MNE_DATA_DIR, verbose=False)
        return subj, None
    except Exception as e:
        return subj, str(e)

completed = 0
n_failed = 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(_fetch_one, s): s for s in todo}
    for fut in as_completed(futures):
        completed += 1
        subj, err = fut.result()
        if err is not None:
            n_failed += 1
            print(f'  subject {subj:3d}: failed ({err[:80]})')
        if completed % 10 == 0 or completed == len(todo):
            print(f'  [{completed:3d}/{len(todo)}] last completed: subject {subj}')

still_missing = [s for s in ALL_SUBJECTS_MI if not subject_fully_cached(s, MI_RUNS)]
if still_missing:
    print(f'WARNING: {len(still_missing)} subjects still incomplete: {still_missing}')
    print('Re-run this cell to resume; partial files are skipped automatically.')
else:
    print(f'All {len(ALL_SUBJECTS_MI)} PhysioNet MI subjects ready.')

PhysioNet MI: cached 105/105 subjects. Downloading 0 in parallel...
All 105 PhysioNet MI subjects ready.


## 4b. Download BCIC-2B data

In [5]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

from moabb.datasets import BNCI2014_004
import moabb
moabb.set_log_level('WARNING')

ds = BNCI2014_004()
print('Downloading BCIC-2B (BNCI2014_004)...')
try:
    ds.download(subject_list=list(range(1, 10)))
    print('BCIC-2B download complete.')
except Exception as e:
    # MOABB sometimes raises on partial cache; data may still be usable
    print(f'MOABB download reported: {e}')
    print('Attempting to load subject 1 to verify cache...')
    try:
        _ = ds.get_data(subjects=[1])
        print('Subject 1 loaded OK — cache is usable.')
    except Exception as e2:
        print(f'WARNING: could not load subject 1: {e2}')

BCIC-2B download complete.


## 4c. Download Sleep-EDFx data

In [6]:
import os, mne
from concurrent.futures import ThreadPoolExecutor, as_completed

mne.set_log_level('WARNING')
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Resume-aware: pooch hash-checks every file before downloading and skips
# files that are already on Drive — re-running this cell only re-attempts
# the missing/failed ones. Each night is fetched independently so a
# night-2 throttle doesn't cause us to re-download night 1.
_UNAVAILABLE = {39, 68, 69, 78, 79}
ALL_SUBJECTS_SLEEP = [s for s in range(0, 83) if s not in _UNAVAILABLE]

N_WORKERS = 4  # PhysioNet rate-limits aggressive parallelism; 4 is the sweet spot
print(f'Sleep-EDFx: fetching {len(ALL_SUBJECTS_SLEEP)} subjects with '
      f'{N_WORKERS} parallel workers (cached subjects skip instantly)...')

def _fetch_one(subj):
    """Fetch night 1 and night 2 independently. Returns (subj, status)."""
    n1_ok = n2_ok = False
    try:
        mne.datasets.sleep_physionet.age.fetch_data(
            subjects=[subj], recording=[1], path=MNE_DATA_DIR, verbose=False
        )
        n1_ok = True
    except Exception:
        pass
    try:
        mne.datasets.sleep_physionet.age.fetch_data(
            subjects=[subj], recording=[2], path=MNE_DATA_DIR, verbose=False
        )
        n2_ok = True
    except Exception:
        pass
    if n1_ok and n2_ok:
        return subj, 'both'
    if n1_ok:
        return subj, 'night1_only'
    if n2_ok:
        return subj, 'night2_only'
    return subj, 'failed'

n_done = n_failed = 0
completed = 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(_fetch_one, s): s for s in ALL_SUBJECTS_SLEEP}
    for fut in as_completed(futures):
        completed += 1
        subj, status = fut.result()
        if status == 'failed':
            n_failed += 1
        else:
            n_done += 1
        if completed % 5 == 0 or completed == len(ALL_SUBJECTS_SLEEP):
            print(f'  [{completed:3d}/{len(ALL_SUBJECTS_SLEEP)}] '
                  f'ok={n_done}  failed={n_failed}  (last: subject {subj}, {status})')

print(f'\nSleep-EDFx: {n_done} subjects cached, {n_failed} failed. '
      f'Re-run this cell to retry failures (already-cached files will skip).')

Sleep-EDFx: fetching 78 subjects with 4 parallel workers (cached subjects skip instantly)...
  [  5/78] ok=5  failed=0  (last: subject 4, both)
  [ 10/78] ok=10  failed=0  (last: subject 9, both)
  [ 15/78] ok=15  failed=0  (last: subject 12, both)
  [ 20/78] ok=20  failed=0  (last: subject 19, both)
  [ 25/78] ok=25  failed=0  (last: subject 25, both)
  [ 30/78] ok=30  failed=0  (last: subject 29, both)
  [ 35/78] ok=35  failed=0  (last: subject 34, both)
  [ 40/78] ok=40  failed=0  (last: subject 41, both)
  [ 45/78] ok=45  failed=0  (last: subject 47, both)
  [ 50/78] ok=50  failed=0  (last: subject 51, both)
  [ 55/78] ok=55  failed=0  (last: subject 55, both)
  [ 60/78] ok=60  failed=0  (last: subject 60, both)
  [ 65/78] ok=65  failed=0  (last: subject 64, both)
  [ 70/78] ok=70  failed=0  (last: subject 73, both)
  [ 75/78] ok=75  failed=0  (last: subject 76, both)
  [ 78/78] ok=78  failed=0  (last: subject 81, both)

Sleep-EDFx: 78 subjects cached, 0 failed. Re-run this cell to

## 5. Build preprocessing caches (CPU)

After raw data is downloaded, run each loader once to produce a cached
`.npz` of preprocessed arrays under `{DRIVE_ROOT}/cache/`. Each subsequent
pretrain or eval run loads from this cache in ~seconds instead of
re-doing ~15 minutes of bandpass + resample + epoch + normalize per dataset.

Cache key is a hash of the preprocessing config — change any preprocessing
knob (sample rate, bandpass, epoch length, etc.) and a fresh cache will
be built automatically; the old file remains until you delete it.

Each subsection below is independent and skippable. Re-running a cell with
an already-built cache prints `[cache] loading ...` and exits in seconds.

### 5a. PhysioNet MI cache (pretrain mode — 105 subjects)

In [7]:
import os, sys
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')

from config import Config
from datasets.physionet_mi import PhysioNetMI, EXCLUDED_SUBJECTS

cfg = Config()
ALL_MI = [s for s in range(1, 110) if s not in EXCLUDED_SUBJECTS]
print(f'Building PhysioNet MI cache for {len(ALL_MI)} subjects (pretrain mode)...')
X, y, ch_pos, ch_names = PhysioNetMI(subjects=ALL_MI, cfg=cfg,
                                     mode='pretrain', verbose=True).load()
print(f'\nX: {X.shape}  y: {y.shape}  ch_pos: {ch_pos.shape}  ch_names: {len(ch_names)}')

Building PhysioNet MI cache for 105 subjects (pretrain mode)...
[cache] loading physionet_mi/pretrain from physionet_mi_pretrain_1bfb941d013e1e1e.npz

X: (9408, 64, 800)  y: (9408,)  ch_pos: (64, 3)  ch_names: 64


### 5b. PhysioNet MI eval cache (105 per-subject files)

probe.py loads PhysioNet MI one subject at a time, so we build one cache
file per subject. RAM stays low; the cache key includes the subject list
so probe.py's `subjects=[s]` calls hit these files directly.

In [8]:
import os, sys
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')

from config import Config
from datasets.physionet_mi import PhysioNetMI, EXCLUDED_SUBJECTS

# Eval mode is consumed by probe.py one subject at a time, so we build
# per-subject caches (low RAM, matches probe.py's access pattern).
cfg = Config()
ALL_MI = [s for s in range(1, 110) if s not in EXCLUDED_SUBJECTS]
print(f'Building PhysioNet MI eval cache per-subject ({len(ALL_MI)} subjects)...')
for k, s in enumerate(ALL_MI, start=1):
    PhysioNetMI(subjects=[s], cfg=cfg, mode='eval', verbose=False).load()
    if k % 10 == 0 or k == len(ALL_MI):
        print(f'  [{k:3d}/{len(ALL_MI)}] last cached: subject {s}')
print('PhysioNet MI eval caches done.')

Building PhysioNet MI cache for 105 subjects (eval mode)...
[cache] loading physionet_mi/eval from physionet_mi_eval_a0044464e0ee3ec7.npz
X: (9408, 64, 800)  y: (9408,)


### 5c. BCIC-2B cache (eval mode — 9 subjects)

In [9]:
import os, sys
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')

from config import Config
from datasets.bcic_2b import BCIC2B, ALL_SUBJECTS

cfg = Config()
print(f'Building BCIC-2B cache for {len(ALL_SUBJECTS)} subjects (eval mode)...')
X, y, ch_pos, ch_names = BCIC2B(subjects=list(ALL_SUBJECTS), cfg=cfg,
                                mode='eval', verbose=False).load()
print(f'X: {X.shape}  y: {y.shape}  ch_pos: {ch_pos.shape}')

Building BCIC-2B cache for 9 subjects (eval mode)...
[cache] loading bcic_2b/eval from bcic_2b_eval_bab5e6065c3c41f0.npz
X: (6520, 3, 800)  y: (6520,)  ch_pos: (3, 3)


### 5d. Sleep-EDFx eval cache (per-subject)

Sleep-EDFx is the largest dataset (~78 subjects × 2 nights). A full-corpus
build OOMs Colab free tier. probe.py only ever loads one subject at a time
(see `_load_one_subject`), so we build one cache file per subject here —
RAM stays at one subject's worth. Expect ~20–40 min total.

In [ ]:
import os, sys, gc
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')

from config import Config
from datasets.sleep_edfx import SleepEDFx, ALL_SUBJECTS, _MISSING_RECORDINGS

# Sleep-EDFx is large — the full-corpus cache OOMs Colab free tier (~12 GB).
# probe.py only ever loads one subject at a time (see _load_one_subject),
# so we build a per-subject cache here. RAM stays at one subject's worth.
cfg = Config()
TODO = list(ALL_SUBJECTS)
print(f'Building Sleep-EDFx eval cache per-subject ({len(TODO)} subjects)...')
ok = skipped = failed = 0
for k, s in enumerate(TODO, start=1):
    try:
        SleepEDFx(subjects=[s], cfg=cfg, mode='eval', verbose=False).load()
        ok += 1
    except RuntimeError as e:
        # No epochs (e.g. subject 13 missing night 2 etc) — _MISSING_RECORDINGS
        # filters internally; if nothing remains the loader raises.
        skipped += 1
        print(f'  subject {s}: skipped ({e})')
    except Exception as e:
        failed += 1
        print(f'  subject {s}: FAILED ({type(e).__name__}: {str(e)[:100]})')
    gc.collect()
    if k % 5 == 0 or k == len(TODO):
        print(f'  [{k:3d}/{len(TODO)}] ok={ok} skipped={skipped} failed={failed}')
print(f'\nSleep-EDFx caches: {ok} ok, {skipped} skipped, {failed} failed.')

Building Sleep-EDFx cache for 78 subjects (eval mode)...
[cache] building sleep_edfx/eval; will save to sleep_edfx_eval_30e7a0bdafc589da.npz


/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:209: RuntimeWarning: Requested recording 1 for subject 36 and/or 52, but it is not available in corpus.
  files = mne.datasets.sleep_physionet.age.fetch_data(
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:209: RuntimeWarning: Requested recording 2 for subject 13, but it is not available in corpus.
  files = mne.datasets.sleep_physionet.age.fetch_data(
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lo

  subject  0 night 1: 19825 epochs


/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")


  subject  0 night 2: 21169 epochs


...

/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")


  subject 36 night 2: 16971 epochs


/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")


  subject 37 night 1: 21344 epochs


/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=False, verbose="WARNING")


## Done

Close this runtime to free CPU resources. Both raw downloads AND preprocessing
caches now live on Drive. Open `colab_pretrain.ipynb` with a **GPU runtime**
to start training — it will read `{DRIVE_ROOT}/cache/` directly and skip the
~15 min preprocessing per variant.